# UAV Object Detection · Detectron2 lab
## 07 · One-stage detection: RetinaNet (and where YOLO fits)
Supports the **YOLO Grid** playground and the *framework vs architecture* exam trap. **Important:** Detectron2 does **not** implement YOLO — YOLO's framework is **Ultralytics**. Detectron2's one-stage detector is **RetinaNet**, so we train that here to demonstrate the single-pass idea, then show the real YOLO on the same data as an optional extra.

In [ ]:
# ============================================================
#  Detectron2 install (Google Colab)  —  enable a GPU runtime:
#  Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU
# ============================================================
# Detectron2 ships no current pre-built wheels, so we compile it from
# source against whatever torch Colab has today. This takes a few minutes.
!python -m pip install -q 'pyyaml==6.0.*'
import torch, subprocess, sys
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

# If the build FAILS because Colab's torch is newer than detectron2 supports,
# pin a known-good torch first, then re-run this cell:
#   !python -m pip install -q torch==2.1.2 torchvision==0.16.2
#   !python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
# After a successful install you may need:  Runtime ▸ Restart session.

try:
    import detectron2
    print("detectron2:", detectron2.__version__, "✓ ready")
except Exception as e:
    print("detectron2 not importable yet — restart the session and re-run.\n", e)


In [ ]:
%%writefile drone_synth.py
"""
drone_synth.py — contextualized synthetic drone (nadir/aerial) imagery.

Generates top-down scenes (parking lots and crop fields) populated with
cars, people and trees, plus COCO-format annotations (bbox + polygon
segmentation). Pure numpy + Pillow, so it runs with no GPU and no
Detectron2 install. The visual language mirrors the HTML study
playgrounds: magenta/blue accents, nadir vehicles, small people from
altitude, cluttered backgrounds.

Categories (COCO ids are 1-indexed):
    1 = car     2 = person     3 = tree

Key entry points:
    make_scene(seed, kind)          -> (PIL.Image, [ann, ...])
    build_coco(n, out_dir, split)   -> path to COCO json (+ images on disk)
    THING_CLASSES                   -> ["car", "person", "tree"]
"""
import json, math, os, random
import numpy as np
from PIL import Image, ImageDraw

THING_CLASSES = ["car", "person", "tree"]
W_DEF, H_DEF = 512, 512

# ---- palette (kept close to the HTML playgrounds) -------------------------
ASPHALT = (157, 162, 148)
FIELD   = (170, 182, 132)
CROP    = (139, 155, 113)
DIRT    = (176, 152, 122)
LANE    = (232, 228, 210)
CAUTION = (200, 150, 40)
CAR_COLORS = [(122, 48, 48), (47, 90, 138), (90, 106, 47), (90, 63, 110), (60, 60, 66)]


def _poly_from_ellipse(cx, cy, rx, ry, n=16):
    return [(cx + rx * math.cos(2 * math.pi * i / n),
             cy + ry * math.sin(2 * math.pi * i / n)) for i in range(n)]


def _rect_poly(x, y, w, h):
    return [(x, y), (x + w, y), (x + w, y + h), (x, y + h)]


def _draw_car(dr, box, color):
    x, y, w, h = box
    pad = min(w, h) * 0.06
    dr.rounded_rectangle([x + pad, y + pad, x + w - pad, y + h - pad],
                         radius=min(w, h) * 0.16, fill=color, outline=(14, 18, 22))
    vertical = h >= w
    if vertical:
        dr.rounded_rectangle([x + w * 0.22, y + h * 0.30, x + w * 0.78, y + h * 0.64],
                             radius=3, fill=(143, 160, 173))
        dr.rounded_rectangle([x + w * 0.26, y + h * 0.12, x + w * 0.74, y + h * 0.24],
                             radius=2, fill=(199, 210, 218))
    else:
        dr.rounded_rectangle([x + w * 0.30, y + h * 0.22, x + w * 0.64, y + h * 0.78],
                             radius=3, fill=(143, 160, 173))
        dr.rounded_rectangle([x + w * 0.12, y + h * 0.26, x + w * 0.24, y + h * 0.74],
                             radius=2, fill=(199, 210, 218))
    return _rect_poly(x + pad, y + pad, w - 2 * pad, h - 2 * pad)


def _draw_person(dr, box):
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    r = min(w, h) * 0.30
    dr.ellipse([cx - r * 1.1, cy + r * 0.1, cx + r * 1.1, cy + r * 0.9], fill=(0, 0, 0, 60))
    dr.ellipse([cx - r * 0.85, cy + r * 0.0, cx + r * 0.85, cy + r * 1.0], fill=(107, 79, 140), outline=(14, 18, 22))
    dr.ellipse([cx - r * 0.55, cy - r * 0.8, cx + r * 0.55, cy + r * 0.3], fill=(202, 162, 122), outline=(14, 18, 22))
    return _poly_from_ellipse(cx, cy, w * 0.42, h * 0.42, 14)


def _draw_tree(dr, box):
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    r = min(w, h) * 0.5
    dr.ellipse([cx - r * 0.95, cy - r * 0.9, cx + r * 0.95, cy + r * 0.95], fill=(79, 122, 67), outline=(47, 74, 40))
    dr.ellipse([cx - r * 0.6, cy - r * 0.55, cx + r * 0.2, cy + r * 0.1], fill=(95, 143, 80))
    return _poly_from_ellipse(cx, cy, r * 0.9, r * 0.9, 16)


def _backdrop(img, dr, kind, rng):
    W, H = img.size
    if kind == "field":
        dr.rectangle([0, 0, W, H], fill=FIELD)
        for yy in range(8, H, 15):
            dr.line([(0, yy), (W, yy)], fill=CROP, width=6)
        # a curved dirt track
        pts = [(0, H * 0.72)]
        for t in range(1, 11):
            pts.append((W * t / 10, H * (0.62 + 0.12 * math.sin(t))))
        dr.line(pts, fill=DIRT, width=18)
    else:
        dr.rectangle([0, 0, W, H], fill=ASPHALT)
        for x in range(40, W - 20, 52):
            dr.line([(x, 14), (x, H * 0.42)], fill=LANE, width=2)
            dr.line([(x, H * 0.58), (x, H - 14)], fill=LANE, width=2)
        for x in range(0, W, 26):  # dashed centre caution line
            dr.line([(x, H * 0.5), (x + 14, H * 0.5)], fill=CAUTION, width=2)


def _overlaps(box, placed, slack=-6):
    x, y, w, h = box
    for (px, py, pw, ph) in placed:
        if (x < px + pw - slack and x + w > px + slack and
                y < py + ph - slack and y + h > py + slack):
            return True
    return False


def make_scene(seed=0, kind=None, size=(W_DEF, H_DEF)):
    """Return (PIL.Image RGB, annotations). Each annotation:
       {category_id, bbox:[x,y,w,h], segmentation:[[...]], area, iscrowd}."""
    rng = random.Random(seed)
    if kind is None:
        kind = rng.choice(["lot", "field"])
    W, H = size
    img = Image.new("RGB", (W, H))
    dr = ImageDraw.Draw(img, "RGBA")
    _backdrop(img, dr, kind, rng)

    anns, placed = [], []
    n_cars = rng.randint(3, 6)
    n_people = rng.randint(2, 5)
    n_trees = rng.randint(1, 3)

    def place(cat, wr, hr, tries=40):
        for _ in range(tries):
            w = rng.uniform(*wr); h = rng.uniform(*hr)
            if rng.random() < 0.5 and cat == 1:  # some cars rotated to horizontal
                w, h = h, w
            x = rng.uniform(6, W - w - 6); y = rng.uniform(6, H - h - 6)
            box = (x, y, w, h)
            if not _overlaps(box, placed):
                placed.append(box); return box
        return None

    for _ in range(n_cars):
        b = place(1, (60, 96), (86, 120))
        if b:
            poly = _draw_car(dr, b, rng.choice(CAR_COLORS))
            anns.append(_ann(1, b, poly))
    for _ in range(n_people):  # small from altitude
        b = place(2, (14, 26), (18, 34))
        if b:
            poly = _draw_person(dr, b)
            anns.append(_ann(2, b, poly))
    for _ in range(n_trees):
        b = place(3, (54, 90), (54, 90))
        if b:
            poly = _draw_tree(dr, b)
            anns.append(_ann(3, b, poly))
    return img, anns


def _ann(cat, box, poly):
    x, y, w, h = box
    seg = [round(float(v), 1) for pt in poly for v in pt]
    return {"category_id": cat, "bbox": [round(float(x), 1), round(float(y), 1),
            round(float(w), 1), round(float(h), 1)],
            "segmentation": [seg], "area": float(w * h), "iscrowd": 0}


def build_coco(n=40, out_dir="drone_coco", split="train", size=(W_DEF, H_DEF), seed0=0):
    """Write n images + one COCO json. Returns (json_path, img_dir)."""
    img_dir = os.path.join(out_dir, split)
    os.makedirs(img_dir, exist_ok=True)
    images, annotations = [], []
    ann_id = 1
    for i in range(n):
        img, anns = make_scene(seed=seed0 + i, size=size)
        fn = f"{split}_{i:04d}.png"
        img.save(os.path.join(img_dir, fn))
        images.append({"id": i, "file_name": fn, "width": size[0], "height": size[1]})
        for a in anns:
            a = dict(a); a["id"] = ann_id; a["image_id"] = i
            annotations.append(a); ann_id += 1
    coco = {"images": images, "annotations": annotations,
            "categories": [{"id": i + 1, "name": c} for i, c in enumerate(THING_CLASSES)]}
    jp = os.path.join(out_dir, f"{split}.json")
    with open(jp, "w") as f:
        json.dump(coco, f)
    return jp, img_dir


# convenience: numpy image + boxes for the metric notebooks
def scene_arrays(seed=0, kind="lot", size=(W_DEF, H_DEF)):
    img, anns = make_scene(seed, kind, size)
    boxes = np.array([a["bbox"] for a in anns], dtype=float)  # xywh
    labels = np.array([a["category_id"] for a in anns], dtype=int)
    return np.asarray(img), boxes, labels


In [ ]:
from drone_synth import build_coco, THING_CLASSES
train_json, train_dir = build_coco(n=60, out_dir='drone_coco', split='train', seed0=0)
val_json,   val_dir   = build_coco(n=16, out_dir='drone_coco', split='val',   seed0=5000)
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetCatalog, MetadataCatalog
for n in ['drone_train','drone_val']:
    if n in DatasetCatalog.list(): DatasetCatalog.remove(n)
register_coco_instances('drone_train', {}, train_json, train_dir)
register_coco_instances('drone_val',   {}, val_json,   val_dir)
for n in ['drone_train','drone_val']: MetadataCatalog.get(n).thing_classes = THING_CLASSES

### Train RetinaNet (one-stage, dense prediction — no proposal step)

In [ ]:
import torch, os
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file('COCO-Detection/retinanet_R_50_FPN_3x.yaml'))
cfg.DATASETS.TRAIN = ('drone_train',); cfg.DATASETS.TEST = ('drone_val',)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url('COCO-Detection/retinanet_R_50_FPN_3x.yaml')
cfg.SOLVER.IMS_PER_BATCH = 2; cfg.SOLVER.BASE_LR = 0.0025
cfg.SOLVER.MAX_ITER = 400; cfg.SOLVER.STEPS = []
cfg.MODEL.RETINANET.NUM_CLASSES = len(THING_CLASSES)
cfg.MODEL.DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.OUTPUT_DIR = 'out_retina'; os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
trainer = DefaultTrainer(cfg); trainer.resume_or_load(resume=False); trainer.train()

### Evaluate + visualise

In [ ]:
import cv2, random, matplotlib.pyplot as plt, os
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR,'model_final.pth')
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = 0.5
ev = COCOEvaluator('drone_val', output_dir=cfg.OUTPUT_DIR)
print(inference_on_dataset(DefaultPredictor(cfg).model, build_detection_test_loader(cfg,'drone_val'), ev))
pred = DefaultPredictor(cfg); d = random.choice(DatasetCatalog.get('drone_val')); im = cv2.imread(d['file_name'])
v = Visualizer(im[:,:,::-1], MetadataCatalog.get('drone_val'), scale=1.0)
vis = v.draw_instance_predictions(pred(im)['instances'].to('cpu'))
plt.figure(figsize=(7,7)); plt.imshow(vis.get_image()); plt.axis('off'); plt.title('RetinaNet (one-stage) on drone scene'); plt.show()

### The YOLO grid, conceptually
YOLO tiles the image into an **S×S** grid; the cell containing an object's centre predicts it, emitting one **S×S×(B·5+C)** tensor in a single pass. Shrink S and two centres can land in one cell — the classic small/clustered failure a UAV hits from altitude.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from drone_synth import scene_arrays
img, boxes, labels = scene_arrays(seed=3, kind='field')
def yolo_grid(S, B=2, C=3):
    cell = img.shape[0]/S
    owners = {}
    for (x,y,w,h) in boxes:
        gx,gy = int((x+w/2)//cell), int((y+h/2)//cell)
        owners.setdefault((gx,gy),0); owners[(gx,gy)] += 1
    clashes = sum(v-1 for v in owners.values() if v>1)
    return cell, owners, clashes, S*S*(B*5+C)
for S in [13,7,4]:
    cell,own,cl,t = yolo_grid(S)
    print(f'S={S:2d}: tensor {S}x{S}x(2·5+3)={t:5d} values, cell collisions={cl}')

In [ ]:
S=7; cell=img.shape[0]/S
fig,ax=plt.subplots(figsize=(6,6)); ax.imshow(img); ax.axis('off')
import matplotlib.patches as mp
for i in range(1,S): ax.axhline(i*cell,color='w',lw=.6,alpha=.5); ax.axvline(i*cell,color='w',lw=.6,alpha=.5)
for (x,y,w,h) in boxes:
    ax.plot(x+w/2,y+h/2,'o',color='#B3157E')
    gx,gy=int((x+w/2)//cell),int((y+h/2)//cell)
    ax.add_patch(mp.Rectangle((gx*cell,gy*cell),cell,cell,color='#B3157E',alpha=.15))
ax.set_title(f'YOLO {S}×{S} grid — magenta = owning cell'); plt.show()

### Optional: the *real* YOLO via Ultralytics
This is the honest comparison — a genuine single-pass YOLO, trained on the same synthetic drone data exported to YOLO label format.

In [ ]:
# OPTIONAL — installs Ultralytics (independent of Detectron2)
!python -m pip install -q ultralytics
import os, json, glob, yaml
def coco_to_yolo(coco_json, img_dir, out_root, split):
    coco = json.load(open(coco_json)); os.makedirs(f'{out_root}/images/{split}',exist_ok=True); os.makedirs(f'{out_root}/labels/{split}',exist_ok=True)
    imgs={i['id']:i for i in coco['images']}
    from collections import defaultdict; per=defaultdict(list)
    for a in coco['annotations']: per[a['image_id']].append(a)
    import shutil
    for iid,im in imgs.items():
        shutil.copy(os.path.join(img_dir,im['file_name']), f"{out_root}/images/{split}/{im['file_name']}")
        W,H=im['width'],im['height']; lines=[]
        for a in per[iid]:
            x,y,w,h=a['bbox']; xc=(x+w/2)/W; yc=(y+h/2)/H
            lines.append(f"{a['category_id']-1} {xc:.5f} {yc:.5f} {w/W:.5f} {h/H:.5f}")
        open(f"{out_root}/labels/{split}/{im['file_name'].replace('.png','.txt')}",'w').write('\n'.join(lines))
coco_to_yolo(train_json, train_dir, 'yolo_ds', 'train')
coco_to_yolo(val_json,   val_dir,   'yolo_ds', 'val')
yaml.safe_dump({'path':os.path.abspath('yolo_ds'),'train':'images/train','val':'images/val',
                'names':{0:'car',1:'person',2:'tree'}}, open('drone.yaml','w'))
from ultralytics import YOLO
m = YOLO('yolov8n.pt'); m.train(data='drone.yaml', epochs=15, imgsz=512, verbose=False)
print('YOLOv8 trained on the same synthetic drone data')